# Random Forest benchmark - Letter và Digits

Benchmark độc lập cho Letter Recognition và Handwritten Digits. Hai bộ dữ
liệu dùng đúng protocol `test_size=0.20`, `random_state=42`, stratification;
Letter dùng canonical split đã chuẩn bị. Accelerator khuyến nghị: **None (CPU)**.


In [ ]:
import json
import platform
import subprocess
import time
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.datasets import load_digits
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None

    def display(value):
        print(value)

sns.set_theme(style="whitegrid")

from sklearn.ensemble import RandomForestClassifier


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
KAGGLE_WORKING = Path("/kaggle/working")
RUN_ROOT = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
FIGURES_DIR = RUN_ROOT / "figures"
RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
for directory in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LETTER_FEATURES = [
    "x_box", "y_box", "width", "high", "onpix", "x_bar", "y_bar", "x2bar",
    "y2bar", "xybar", "x2ybr", "xy2br", "x_ege", "xegvy", "y_ege", "yegvx",
]


def detect_hardware():
    hardware = {
        "environment": "kaggle" if KAGGLE_WORKING.exists() else "local",
        "platform": platform.platform(),
        "processor": platform.processor() or platform.machine(),
        "logical_cpu_count": __import__("os").cpu_count(),
        "gpu_available": False,
        "gpus": [],
        "model_compute_device": "cpu",
    }
    try:
        output = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            check=True,
            text=True,
            timeout=10,
        ).stdout.strip()
        for line in output.splitlines():
            name, memory_mb, driver = [part.strip() for part in line.split(",", maxsplit=2)]
            hardware["gpus"].append(
                {"name": name, "memory_mb": int(memory_mb), "driver_version": driver}
            )
        hardware["gpu_available"] = bool(hardware["gpus"])
    except (FileNotFoundError, subprocess.SubprocessError, ValueError):
        pass
    return hardware


hardware = detect_hardware()
print({"python": platform.python_version(), "sklearn": sklearn.__version__})
print("Hardware:", hardware)

EXPERIMENT_ID = "rf_letter_digits_benchmark"
MODEL_NAME = "Random Forest"
MODEL_CONFIG = {'n_estimators': 300, 'criterion': 'gini', 'random_state': 42, 'n_jobs': -1, 'scaling': False}


## Preprocessing

Random Forest không cần scaling. Mọi transformer chỉ được fit trên tập train thông qua pipeline,
tránh data leakage.


In [ ]:
def find_letter_split():
    roots = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
    checked = set()
    for root in roots:
        if not root.exists():
            continue
        for train_path in root.rglob("train.csv"):
            test_path = train_path.with_name("test.csv")
            key = str(train_path.resolve())
            if key in checked or not test_path.exists():
                continue
            checked.add(key)
            try:
                columns = set(pd.read_csv(train_path, nrows=2).columns)
                test_columns = set(pd.read_csv(test_path, nrows=2).columns)
            except (OSError, pd.errors.ParserError):
                continue
            required = set(LETTER_FEATURES + ["letter"])
            if required.issubset(columns) and required.issubset(test_columns):
                return train_path, test_path
    raise FileNotFoundError(
        "Không tìm thấy canonical Letter train.csv/test.csv. "
        "Trên Kaggle, hãy Add Input dataset chứa hai file processed này."
    )


def load_benchmark_datasets():
    started = time.perf_counter()
    letter_train_path, letter_test_path = find_letter_split()
    letter_train = pd.read_csv(letter_train_path)
    letter_test = pd.read_csv(letter_test_path)

    digits = load_digits(as_frame=True)
    digits_frame = digits.frame.rename(columns={"target": "label"})
    digits_train, digits_test = train_test_split(
        digits_frame,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=digits_frame["label"],
    )
    datasets = {
        "letter_recognition": {
            "X_train": letter_train[LETTER_FEATURES],
            "y_train": letter_train["letter"],
            "X_test": letter_test[LETTER_FEATURES],
            "y_test": letter_test["letter"],
            "source": str(letter_train_path.parent),
        },
        "handwritten_digits": {
            "X_train": digits_train.drop(columns="label"),
            "y_train": digits_train["label"].astype("int64"),
            "X_test": digits_test.drop(columns="label"),
            "y_test": digits_test["label"].astype("int64"),
            "source": "sklearn.datasets.load_digits",
        },
    }
    return datasets, time.perf_counter() - started


datasets, data_loading_seconds = load_benchmark_datasets()
for dataset_name, parts in datasets.items():
    print(
        dataset_name,
        "train/test:",
        parts["X_train"].shape,
        parts["X_test"].shape,
        "source:",
        parts["source"],
    )


In [ ]:
def make_estimator():

    return RandomForestClassifier(
        n_estimators=300,
        criterion="gini",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def evaluate_dataset(dataset_name, parts):
    estimator = make_estimator()
    fit_started = time.perf_counter()
    estimator.fit(parts["X_train"], parts["y_train"])
    training_seconds = time.perf_counter() - fit_started

    predict_started = time.perf_counter()
    prediction = estimator.predict(parts["X_test"])
    prediction_seconds = time.perf_counter() - predict_started
    train_prediction = estimator.predict(parts["X_train"])

    train_accuracy = accuracy_score(parts["y_train"], train_prediction)
    test_accuracy = accuracy_score(parts["y_test"], prediction)
    metrics = {
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
        "error_rate": 1.0 - test_accuracy,
        "precision_macro": precision_score(
            parts["y_test"], prediction, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            parts["y_test"], prediction, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(parts["y_test"], prediction, average="macro"),
        "generalization_gap": train_accuracy - test_accuracy,
        "training_seconds": training_seconds,
        "prediction_seconds": prediction_seconds,
        "train_samples": len(parts["X_train"]),
        "test_samples": len(parts["X_test"]),
        "features": parts["X_train"].shape[1],
    }

    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay.from_predictions(
        parts["y_test"], prediction, cmap="Blues", colorbar=False, values_format="d", ax=ax
    )
    ax.set_title(f"{dataset_name} - {MODEL_NAME}")
    fig.tight_layout()
    figure_path = FIGURES_DIR / f"{EXPERIMENT_ID}__{dataset_name}__confusion_matrix.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()

    model_path = MODELS_DIR / f"{EXPERIMENT_ID}__{dataset_name}.joblib"
    joblib.dump(estimator, model_path)
    return estimator, metrics, figure_path, model_path


pipeline_started = time.perf_counter()
evaluations = {}
artifact_paths = []
for dataset_name, parts in datasets.items():
    _, metrics, figure_path, model_path = evaluate_dataset(dataset_name, parts)
    evaluations[dataset_name] = metrics
    artifact_paths.extend([figure_path, model_path])

summary = pd.DataFrame(evaluations).T
display(summary.round(4))


## Kết quả

So sánh accuracy, macro-F1, train-test gap và thời gian với các model khác ở notebook tổng hợp.


In [ ]:
summary_path = RESULTS_DIR / f"{EXPERIMENT_ID}__summary.csv"
summary.reset_index(names="dataset").to_csv(summary_path, index=False)
result_path = RESULTS_DIR / f"{EXPERIMENT_ID}.json"
pipeline_seconds = time.perf_counter() - pipeline_started
result = {
    "schema_version": "1.0",
    "experiment_id": EXPERIMENT_ID,
    "model": MODEL_NAME,
    "model_config": MODEL_CONFIG,
    "split": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratify": True,
        "letter_split": "canonical preprocessed train/test",
    },
    "data_loading_seconds": data_loading_seconds,
    "pipeline_seconds": pipeline_seconds,
    "hardware": hardware,
    "datasets": evaluations,
    "notes": "All sklearn estimators compute on CPU; preprocessing is fit on train only.",
    "created_at_utc": datetime.now(UTC).isoformat(),
}
result_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
artifact_paths.extend([summary_path, result_path])

archive_path = RUN_ROOT / f"{EXPERIMENT_ID}__outputs.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for artifact_path in artifact_paths:
        archive.write(artifact_path, artifact_path.relative_to(RUN_ROOT))

print(f"Created ZIP ({archive_path.stat().st_size / 1024**2:.1f} MB): {archive_path}")
if FileLink is not None:
    display(FileLink(str(archive_path)))
